# Final Project Evaluation: Counterfactual Medical Image Generation

This notebook demonstrates the complete diagnostic pipeline:
1. **Classification** with MC Dropout for uncertainty.
2. **Counterfactual Generation** using CycleGAN.
3. **Explainability** via Grad-CAM and Difference Mapping.

In [ ]:
import os
import sys
import torch
from PIL import Image
import matplotlib.pyplot as plt

# Add project root to path
sys.path.append('..')

from src.utils.inference_engine import InferenceEngine
from src.utils.data_loader import get_dataloaders

## 1. Initialize Inference Engine
We load the best classifier checkpoint and the CycleGAN generator (Epoch 20).

In [ ]:
classifier_path = '../src/models/best_classifier.pth'
cyclegan_path = '../src/models/cyclegan/G_BA.pth'
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

engine = InferenceEngine(classifier_path, cyclegan_path, device=device)
print("Engine Ready!")

## 2. Load Sample Case
We select a sample Pneumonia case from the test set.

In [ ]:
test_dir = '../data/chest_xray/test/PNEUMONIA'
sample_files = [f for f in os.listdir(test_dir) if f.lower().endswith('.jpeg')]
img_path = os.path.join(test_dir, sample_files[0])
img = Image.open(img_path).convert('RGB')

plt.imshow(img)
plt.title("Original X-Ray (Pneumonia Case)")
plt.axis('off')
plt.show()

## 3. Run Pipeline Inference
Executing the full 3-stage pipeline.

In [ ]:
results = engine.run_inference(img, mc_samples=10)

print(f"Prediction: {results['label']}")
print(f"Confidence: {results['confidence']:.2%}")
print(f"Uncertainty: ±{results['uncertainty']:.4f}")

## 4. Visualize Explainability Outputs
Comparing Grad-CAM attention with Counterfactual changes.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(results['gradcam_img'])
axes[1].set_title("Grad-CAM (Attention)")
axes[1].axis('off')

axes[2].imshow(results['counterfactual_img'])
axes[2].set_title("Counterfactual (Healthy)")
axes[2].axis('off')

axes[3].imshow(results['diff_map'])
axes[3].set_title("Difference Map (Pathology)")
axes[3].axis('off')

plt.tight_layout()
plt.show()